# Faster R-CNN（PASCAL VOC 2007を用いた物体検出）

---
## 目的
物体検出の代表的な二段階（two-stage）手法であるFaster R-CNNを，PASCAL VOC 2007データセットを用いて実装する．SSD（`ssd.ipynb`参照）は，1回のネットワーク推論で密に配置したデフォルトボックスから直接検出結果を得る「単一段階（single-shot）」の手法であったのに対し，Faster R-CNNは，まず「物体らしさ」の高い領域候補（Region Proposal）をRegion Proposal Network（RPN）によって絞り込み，次にその候補領域（Proposal）ごとにクラス分類とボックス回帰を行う「二段階（two-stage）」の手法である．RPNと第2段階の検出ヘッドが，特徴マップを共有しながらどのように連携するかを理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
get_ipython().system('pip install -q torchmetrics')

import os
import zipfile
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import box_iou, box_convert, nms, roi_align
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して矩形領域（Bounding Box）が付与されたデータセットで，学習用（trainval）5011枚，評価用（test）4952枚の画像から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCDetection`で読み込みます．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# 背景（物体なし）を0番として，クラスラベルは1〜20を割り当てる
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}
n_class = len(VOC_CLASSES) + 1  # 背景を含めたクラス数
IMG_SIZE = 320

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    """VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する"""
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    """VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景含む）:', n_class)

## バックボーン（ResNet-C4）
原論文のFaster R-CNN + ResNetの構成（いわゆる「ResNet-C4」）にならい，事前学習済みResNet50を，特徴抽出を担う前半部分（`conv1`〜`layer3`，出力ストライド16．これを**C4**と呼ぶ）と，第2段階の検出ヘッドで再利用する後半部分（`layer4`，出力ストライド32．これを**C5**と呼ぶ）に分割します．

- **C4**（`layer3`の出力）は，RPNと後段のROIAlignの両方で共有される単一の特徴マップとして使用します．
- **C5**（`layer4`）は捨てずに，ROIAlignでプーリングした各Proposalの特徴を，第2段階の分類・回帰ヘッドの一部として処理する畳み込み層として再利用します（後述の`ROIHead`を参照）．

これは，複数解像度の特徴マップをすべて検出に使うSSD（`ssd.ipynb`参照）とは対照的に，単一の特徴マップ（C4）だけを検出の土台に使う設計です．

In [ ]:
class FasterRCNNBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3)  # -> C4, stride 16, channels 1024
        self.layer4 = resnet.layer4  # -> C5, stride 32, channels 2048（ROIヘッドで再利用）
        self.out_channels = 1024
        self.stride = 16

    def forward(self, x):
        return self.stem(x)  # RPN・ROIAlignで共有するC4特徴マップのみを返す


backbone_test = FasterRCNNBackbone().to(device)
with torch.no_grad():
    feat = backbone_test(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device))
print('C4特徴マップの形状:', feat.shape)  # (1, 1024, IMG_SIZE/16, IMG_SIZE/16)

## アンカー（Anchor）の生成
RPNは，C4特徴マップの各セル（格子点）に，あらかじめ複数の大きさ・アスペクト比を持つ矩形「アンカー」を敷き詰めておき，各アンカーに対して「物体か背景か」と「アンカーからのずれ」を予測します．本ノートブックでは，3つのスケール（$128^2, 256^2, 512^2$ピクセル）× 3つのアスペクト比（$1:2, 1:1, 2:1$）＝9種類のアンカーを各セルに配置します．これは，SSDのデフォルトボックス生成（`ssd.ipynb`参照）と同様の考え方ですが，特徴マップが1つだけなのでスケールをレベルごとに変える必要がなく，実装がより単純です．

なお，本ノートブックでは計算量を抑えるため入力画像を$320\times320$に縮小しているため，$512^2$のような大きいアンカーは画像からはみ出しやすくなります．これは`decode_boxes`で画像範囲にクリップして扱います．

In [ ]:
ANCHOR_SCALES = [128, 256, 512]
ANCHOR_RATIOS = [0.5, 1.0, 2.0]
FEATURE_STRIDE = 16


def generate_anchors(feature_size, stride=FEATURE_STRIDE, scales=ANCHOR_SCALES, ratios=ANCHOR_RATIOS):
    """C4特徴マップの各セルに対して，複数スケール・アスペクト比のアンカーを配置する（xyxy，ピクセル座標）"""
    anchor_list = []
    for i in range(feature_size):
        for j in range(feature_size):
            cx = (j + 0.5) * stride
            cy = (i + 0.5) * stride
            for scale in scales:
                for ratio in ratios:
                    w = scale * (ratio ** 0.5)
                    h = scale / (ratio ** 0.5)
                    anchor_list.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
    return torch.tensor(anchor_list, dtype=torch.float32)


FEATURE_SIZE = IMG_SIZE // FEATURE_STRIDE
num_anchors_per_cell = len(ANCHOR_SCALES) * len(ANCHOR_RATIOS)
anchors = generate_anchors(FEATURE_SIZE).to(device)  # (num_anchors, 4)
print('アンカー総数:', anchors.shape[0], '=', FEATURE_SIZE, 'x', FEATURE_SIZE, 'x', num_anchors_per_cell)

### ボックスのオフセットのエンコード・デコード
RPN・ROIヘッドのいずれも，ボックス座標を直接回帰するのではなく，基準となるボックス（RPNではアンカー，ROIヘッドではProposal）からの相対的なずれ$(t_x, t_y, t_w, t_h)$を回帰します．中心座標のずれは基準ボックスの幅・高さでスケールし，幅・高さのずれは対数を取ることで，大きさの異なるボックスでもずれのスケールが揃うようにします（原論文と同じパラメータ化）．RPN・ROIヘッドの両方から共通して使うため，基準ボックスを引数として受け取る形にしています．

In [ ]:
def encode_boxes(gt_boxes_xyxy, base_boxes_xyxy):
    """GTボックスを，基準ボックス（アンカー or Proposal）からのオフセット(tx, ty, tw, th)にエンコードする"""
    gt = box_convert(gt_boxes_xyxy, 'xyxy', 'cxcywh')
    base = box_convert(base_boxes_xyxy, 'xyxy', 'cxcywh')
    txty = (gt[:, :2] - base[:, :2]) / base[:, 2:]
    twth = torch.log(gt[:, 2:] / base[:, 2:] + 1e-8)
    return torch.cat([txty, twth], dim=1)


def decode_boxes(deltas, base_boxes_xyxy, img_size=IMG_SIZE):
    """オフセット(tx, ty, tw, th)を基準ボックスに適用し，画像上のボックス座標(xyxy)にデコードする"""
    base = box_convert(base_boxes_xyxy, 'xyxy', 'cxcywh')
    cxcy = deltas[..., :2] * base[:, 2:] + base[:, :2]
    wh = torch.exp(deltas[..., 2:].clamp(max=4.0)) * base[:, 2:]  # expのオーバーフロー防止にクリップ
    boxes_cxcywh = torch.cat([cxcy, wh], dim=-1)
    boxes_xyxy = box_convert(boxes_cxcywh, 'cxcywh', 'xyxy')
    return boxes_xyxy.clamp(min=0, max=img_size)

### RPNの学習ターゲット（アンカーのマッチングとサンプリング）
各アンカーに，IoU（`torchvision.ops.box_iou`で計算）に基づいて次のように正例・負例・無視を割り当てます．

1. GTボックスとのIoUが0.7以上のアンカーは正例（物体）．
2. 各GTボックスに対してIoUが最大のアンカーは，閾値によらず必ず正例にする．
3. すべてのGTボックスとのIoUが0.3未満のアンカーは負例（背景）．
4. どちらでもないアンカーは学習に使わない（無視）．

さらに，1枚の画像に含まれるアンカーの大部分は背景であるため，全アンカーをそのまま学習に使うと背景ばかりが支配的になります．そこでSSDのHard Negative Mining（`ssd.ipynb`参照）とは異なり，正例・負例合わせて`RPN_BATCH_SIZE`個（正例と負例がおよそ1:1）だけをランダムにサンプリングして学習に使用します．

In [ ]:
RPN_POS_IOU, RPN_NEG_IOU = 0.7, 0.3
RPN_BATCH_SIZE, RPN_POS_RATIO = 256, 0.5


def match_anchors(gt_boxes):
    """1枚の画像のGTボックスをアンカーにマッチングし，objectnessラベル(1:正例,0:負例,-1:無視)とオフセットターゲットを作る"""
    num_anchors = anchors.shape[0]
    labels = torch.full((num_anchors,), -1, dtype=torch.long, device=device)

    if gt_boxes.shape[0] == 0:
        labels[:] = 0
        return labels, torch.zeros(num_anchors, 4, device=device)

    ious = box_iou(gt_boxes, anchors)  # (num_gt, num_anchors)
    max_iou_per_anchor, matched_gt_idx = ious.max(dim=0)

    labels[max_iou_per_anchor < RPN_NEG_IOU] = 0
    labels[max_iou_per_anchor >= RPN_POS_IOU] = 1

    # 各GTに対してIoU最大のアンカーは，閾値によらず必ず正例にする
    _, best_anchor_idx = ious.max(dim=1)
    labels[best_anchor_idx] = 1
    matched_gt_idx[best_anchor_idx] = torch.arange(gt_boxes.shape[0], device=device)

    loc_targets = encode_boxes(gt_boxes[matched_gt_idx], anchors)
    return labels, loc_targets


def sample_anchors(labels):
    """正例:負例がおよそ1:1になるよう，学習に使うアンカーをRPN_BATCH_SIZE個だけサンプリングする"""
    pos_idx = (labels == 1).nonzero(as_tuple=True)[0]
    neg_idx = (labels == 0).nonzero(as_tuple=True)[0]

    num_pos = min(pos_idx.numel(), int(RPN_BATCH_SIZE * RPN_POS_RATIO))
    num_neg = min(neg_idx.numel(), RPN_BATCH_SIZE - num_pos)

    pos_idx = pos_idx[torch.randperm(pos_idx.numel(), device=device)[:num_pos]]
    neg_idx = neg_idx[torch.randperm(neg_idx.numel(), device=device)[:num_neg]]

    sample_mask = torch.zeros_like(labels, dtype=torch.bool)
    sample_mask[pos_idx] = True
    sample_mask[neg_idx] = True
    return sample_mask

## RPN（Region Proposal Network）
RPNは，C4特徴マップに対して3x3の畳み込み層を1つ適用したのち，2つの1x1畳み込み（sibling layer）に分岐します．一方はアンカーごとの「物体らしさ」を2クラス（背景/物体）のロジットとして出力し（`nn.CrossEntropyLoss`で扱えるようにするため，1値のBCEではなく2値のロジットとする），もう一方はアンカーごとの4次元のボックスオフセットを出力します．SSDのSSDHead（`ssd.ipynb`参照）と同様，特徴マップの空間方向をそのままアンカー方向に展開する`permute`+`view`で出力を整形します．

In [ ]:
class RPN(nn.Module):
    def __init__(self, in_channels, num_anchors_per_cell):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 512, kernel_size=3, padding=1)
        self.cls_layer = nn.Conv2d(512, num_anchors_per_cell * 2, kernel_size=1)  # 2値(物体/背景)のロジット
        self.reg_layer = nn.Conv2d(512, num_anchors_per_cell * 4, kernel_size=1)

    def forward(self, feature):
        x = F.relu(self.conv(feature))
        batch = feature.size(0)
        cls_logits = self.cls_layer(x).permute(0, 2, 3, 1).contiguous().view(batch, -1, 2)
        box_deltas = self.reg_layer(x).permute(0, 2, 3, 1).contiguous().view(batch, -1, 4)
        return cls_logits, box_deltas

### Proposal（提案領域）の生成
RPNの出力から，実際に第2段階に渡す領域候補（Proposal）を生成します．手順は次の通りです．

1. アンカーごとのオフセットをデコードし，画像上のボックス座標を得る（範囲外は`decode_boxes`内でクリップ済み）．
2. 物体らしさスコアの高い上位`RPN_PRE_NMS_TOP_N`件に絞り込む（NMSの計算量を抑えるため）．
3. `torchvision.ops.nms`でクラス非依存のNMSを適用し，重複するProposalを間引く．
4. スコア上位`RPN_POST_NMS_TOP_N`件だけを最終的なProposalとして残す．

学習時はより多くのProposal候補を残して第2段階の学習データを豊富にし，推論時は計算量を抑えるため少なめに絞り込みます．

In [ ]:
RPN_PRE_NMS_TOP_N = {'train': 2000, 'test': 2000}
RPN_POST_NMS_TOP_N = {'train': 1000, 'test': 300}
RPN_NMS_THRESH = 0.7


def generate_proposals(cls_logits, box_deltas, mode='train'):
    """RPNの出力から，画像ごとのProposal（xyxy，ピクセル座標）のリストを生成する"""
    scores = F.softmax(cls_logits, dim=-1)[..., 1]  # 物体らしさスコア

    proposals_list = []
    for b in range(box_deltas.shape[0]):
        boxes = decode_boxes(box_deltas[b], anchors)

        pre_nms_top_n = min(RPN_PRE_NMS_TOP_N[mode], scores.shape[1])
        top_scores, top_idx = scores[b].topk(pre_nms_top_n)
        top_boxes = boxes[top_idx]

        keep = nms(top_boxes, top_scores, RPN_NMS_THRESH)
        keep = keep[:RPN_POST_NMS_TOP_N[mode]]
        proposals_list.append(top_boxes[keep])

    return proposals_list

## 損失関数（1）RPN Loss
RPNの損失は，サンプリングしたアンカーに対するobjectnessのクロスエントロピー損失と，そのうち正例アンカーのみを対象としたボックス回帰のSmooth L1損失の和です．画像ごとにアンカーのマッチング・サンプリングを行い，バッチ内で平均を取ります．

In [ ]:
def rpn_loss(cls_logits, box_deltas, gt_boxes_list):
    """RPNの学習損失（objectnessのクロスエントロピー + 正例アンカーのSmooth L1）をバッチ全体で計算する"""
    total_cls_loss, total_reg_loss = 0.0, 0.0
    batch_size = len(gt_boxes_list)

    for b, gt_boxes in enumerate(gt_boxes_list):
        labels, loc_targets = match_anchors(gt_boxes)
        sample_mask = sample_anchors(labels)

        cls_loss = F.cross_entropy(cls_logits[b][sample_mask], labels[sample_mask])

        pos_mask = sample_mask & (labels == 1)
        if pos_mask.sum() > 0:
            reg_loss = F.smooth_l1_loss(box_deltas[b][pos_mask], loc_targets[pos_mask], reduction='sum') / sample_mask.sum()
        else:
            reg_loss = torch.zeros((), device=device)

        total_cls_loss = total_cls_loss + cls_loss
        total_reg_loss = total_reg_loss + reg_loss

    return total_cls_loss / batch_size, total_reg_loss / batch_size

### ROIヘッドの学習ターゲット（Proposalのサンプリングとマッチング）
RPNが生成したProposal（および学習を安定させるため，GTボックス自体もProposal候補に加える）に対して，GTボックスとのIoUに基づき正例・負例を決定します．IoUが0.5以上のProposalを正例とし，そのGTのクラスラベルと，Proposalからの相対オフセット（`encode_boxes`）を回帰ターゲットとします．IoUが0.5未満のProposalは背景（クラス0）として扱います．

1枚の画像あたり`ROI_BATCH_SIZE`個のProposalを，正例がおよそ`ROI_POS_RATIO`（25%）になるようサンプリングして学習に使用します．

In [ ]:
ROI_BATCH_SIZE, ROI_POS_RATIO, ROI_POS_IOU = 128, 0.25, 0.5


def sample_proposals(proposals, gt_boxes, gt_labels):
    """Proposalをマッチング・サンプリングし，ROIヘッド学習用の分類・回帰ターゲットを作る"""
    if gt_boxes.shape[0] == 0:
        num_sample = min(ROI_BATCH_SIZE, proposals.shape[0])
        idx = torch.randperm(proposals.shape[0], device=device)[:num_sample]
        sampled = proposals[idx]
        return sampled, torch.zeros(num_sample, dtype=torch.long, device=device), torch.zeros(num_sample, 4, device=device)

    ious = box_iou(gt_boxes, proposals)  # (num_gt, num_proposals)
    max_iou, matched_gt_idx = ious.max(dim=0)

    pos_idx = (max_iou >= ROI_POS_IOU).nonzero(as_tuple=True)[0]
    neg_idx = (max_iou < ROI_POS_IOU).nonzero(as_tuple=True)[0]

    num_pos = min(pos_idx.numel(), int(ROI_BATCH_SIZE * ROI_POS_RATIO))
    num_neg = min(neg_idx.numel(), ROI_BATCH_SIZE - num_pos)
    pos_idx = pos_idx[torch.randperm(pos_idx.numel(), device=device)[:num_pos]]
    neg_idx = neg_idx[torch.randperm(neg_idx.numel(), device=device)[:num_neg]]

    sampled_idx = torch.cat([pos_idx, neg_idx])
    sampled_proposals = proposals[sampled_idx]

    labels = torch.zeros(sampled_idx.numel(), dtype=torch.long, device=device)
    labels[:num_pos] = gt_labels[matched_gt_idx[pos_idx]]

    loc_targets = torch.zeros(sampled_idx.numel(), 4, device=device)
    if num_pos > 0:
        loc_targets[:num_pos] = encode_boxes(gt_boxes[matched_gt_idx[pos_idx]], sampled_proposals[:num_pos])

    return sampled_proposals, labels, loc_targets

## ROIヘッド（第2段階）
`torchvision.ops.roi_align`を用いて，各Proposal領域をC4特徴マップから固定サイズ（7x7）にプーリングします．`spatial_scale`には，C4のストライド（16）の逆数を指定することで，画像座標系のProposalを特徴マップ座標系に変換します．プーリングした特徴は，バックボーンの`layer4`（C5）に通したのち，Global Average Poolingで1つのベクトルに集約し，最後に全結合層でクラススコアとボックス回帰を出力します．

ボックス回帰は，実装を単純化するため全クラス共通の4次元（class-agnostic）とします．原論文はクラスごとに異なる回帰（class-specific，出力次元`n_class * 4`）ですが，本ノートブックでは省略しています（後述の課題で扱います）．

In [ ]:
class ROIHead(nn.Module):
    def __init__(self, layer4, n_class):
        super().__init__()
        self.layer4 = layer4  # backboneのlayer4(C5)を第2段階の特徴抽出に再利用する
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.cls_score = nn.Linear(2048, n_class)
        self.bbox_pred = nn.Linear(2048, 4)  # クラス非依存(class-agnostic)のボックス回帰にして単純化する

    def forward(self, feature, proposals, spatial_scale):
        # proposalsは画像ごとのTensorのリスト。roi_alignが画像インデックスの対応付けを内部で行う
        pooled = roi_align(feature, proposals, output_size=(7, 7), spatial_scale=spatial_scale, aligned=True)
        x = self.layer4(pooled)
        x = self.avgpool(x).flatten(1)
        return self.cls_score(x), self.bbox_pred(x)

## 損失関数（2）ROIヘッド Loss
ROIヘッドの損失は，サンプリングした全Proposal（正例・負例とも）を対象としたクラス分類のクロスエントロピー損失と，正例Proposalのみを対象としたボックス回帰のSmooth L1損失の和です．最終的な学習損失は，このROIヘッドLossとRPN Loss（前述）を単純に合算した1つの総損失として，end-to-endで同時に逆伝播します（原論文の4-step alternating trainingのような段階的な学習は行いません）．

In [ ]:
def roi_loss(cls_scores, box_deltas, labels, loc_targets):
    """ROIヘッドの学習損失（クラス分類のクロスエントロピー + 正例のみのSmooth L1）を計算する"""
    cls_loss = F.cross_entropy(cls_scores, labels)

    pos_mask = labels > 0
    if pos_mask.sum() > 0:
        reg_loss = F.smooth_l1_loss(box_deltas[pos_mask], loc_targets[pos_mask], reduction='sum') / labels.numel()
    else:
        reg_loss = torch.zeros((), device=device)

    return cls_loss, reg_loss

## ネットワークの作成
バックボーン・RPN・ROIヘッドをまとめた`FasterRCNN`を定義します．`backbone.layer4`と`roi_head.layer4`は同じ`nn.Module`インスタンスを共有しているため（C5の再利用），`model.parameters()`で重複してカウントされることはありません．`.to(device)`によりモデルのパラメータを`device`上に配置し，最適化手法にはモーメンタムSGD（学習率0.001，モーメンタム0.9，weight decay 5e-4）を用います．事前学習済みのバックボーンを使用するため，スクラッチ学習のCIFAR100分類モデルよりも小さめの学習率を設定しています．

In [ ]:
class FasterRCNN(nn.Module):
    def __init__(self, n_class, num_anchors_per_cell):
        super().__init__()
        self.backbone = FasterRCNNBackbone()
        self.rpn = RPN(self.backbone.out_channels, num_anchors_per_cell)
        self.roi_head = ROIHead(self.backbone.layer4, n_class)
        self.spatial_scale = 1.0 / self.backbone.stride

    def extract_features(self, images):
        return self.backbone(images)


model = FasterRCNN(n_class=n_class, num_anchors_per_cell=num_anchors_per_cell)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
読み込んだVOC2007データセットと作成したネットワークを用いて学習を行います．1イテレーションあたりの流れは次の通りです．

1. バックボーンでC4特徴マップを抽出し，RPNでobjectnessとボックスオフセットを予測，RPN Lossを計算する．
2. RPNの出力（勾配を切り離したもの）からProposalを生成し，GTボックスを候補に加えたのち，画像ごとにサンプリングしてROIヘッドに入力，ROIヘッド Lossを計算する．
3. RPN LossとROIヘッド Lossを合算した総損失を1回のbackwardで逆伝播する．

ROIヘッドは画像ごとにサンプルするProposal数が異なり，`layer4`を各Proposalに適用する分だけSSDより1イテレーションあたりの計算量が大きいため，ミニバッチサイズはSSDより小さい4とします．エポック数は20とします．

In [ ]:
batch_size = 4
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss, sum_rpn_loss, sum_roi_loss = 0.0, 0.0, 0.0

    for images, targets in train_loader:
        images = images.to(device)
        gt_boxes_list = [t['boxes'].to(device) for t in targets]
        gt_labels_list = [t['labels'].to(device) for t in targets]

        features = model.extract_features(images)

        cls_logits, box_deltas = model.rpn(features)
        rpn_cls_loss, rpn_reg_loss = rpn_loss(cls_logits, box_deltas, gt_boxes_list)

        with torch.no_grad():
            proposals_list = generate_proposals(cls_logits, box_deltas, mode='train')

        roi_cls_loss_total, roi_reg_loss_total = 0.0, 0.0
        for b in range(images.shape[0]):
            # 学習初期はRPNのProposalがGTと重ならないことが多いため，GTボックス自体も候補に加えて安定させる
            if gt_boxes_list[b].shape[0] > 0:
                proposals = torch.cat([proposals_list[b], gt_boxes_list[b]], dim=0)
            else:
                proposals = proposals_list[b]

            sampled_proposals, labels, loc_targets = sample_proposals(proposals, gt_boxes_list[b], gt_labels_list[b])
            cls_scores, deltas = model.roi_head(features[b:b + 1], [sampled_proposals], model.spatial_scale)
            cls_loss, reg_loss = roi_loss(cls_scores, deltas, labels, loc_targets)
            roi_cls_loss_total = roi_cls_loss_total + cls_loss
            roi_reg_loss_total = roi_reg_loss_total + reg_loss

        roi_cls_loss_total = roi_cls_loss_total / images.shape[0]
        roi_reg_loss_total = roi_reg_loss_total / images.shape[0]

        loss = rpn_cls_loss + rpn_reg_loss + roi_cls_loss_total + roi_reg_loss_total

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        sum_rpn_loss += (rpn_cls_loss + rpn_reg_loss).item()
        sum_roi_loss += (roi_cls_loss_total + roi_reg_loss_total).item()

    print("epoch: {}, mean loss: {}, rpn loss: {}, roi loss: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), sum_rpn_loss / len(train_loader), sum_roi_loss / len(train_loader), time() - train_start))

## 推論（Proposal生成・デコード・NMS）
学習したモデルで推論を行うための関数を定義します．バックボーンでC4特徴マップを抽出し，RPNでProposalを生成したのち，ROIヘッドで各Proposalのクラススコアとボックスオフセットを求めます．クラスごとに信頼度が閾値を超えるボックスをデコードし，`torchvision.ops.nms`でクラスごとの重複を除去します．流れ自体はSSD（`ssd.ipynb`の`predict()`）と同様ですが，途中にRPNによるProposal生成という段階が挟まる点が異なります．

In [ ]:
def predict(model, image_tensor, conf_threshold=0.5, nms_threshold=0.45):
    model.eval()
    with torch.no_grad():
        features = model.extract_features(image_tensor.unsqueeze(0).to(device))
        cls_logits, box_deltas = model.rpn(features)
        proposals = generate_proposals(cls_logits, box_deltas, mode='test')[0]

        if proposals.shape[0] == 0:
            return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

        cls_scores, deltas = model.roi_head(features, [proposals], model.spatial_scale)
        scores_all = F.softmax(cls_scores, dim=-1)
        boxes_all = decode_boxes(deltas, proposals)  # class-agnosticなので全クラス共通のボックス

    result_boxes, result_scores, result_labels = [], [], []
    for class_idx in range(1, n_class):  # 背景（クラス0）は除外
        class_scores = scores_all[:, class_idx]
        mask = class_scores > conf_threshold
        if mask.sum() == 0:
            continue
        class_boxes = boxes_all[mask]
        class_scores = class_scores[mask]

        keep = nms(class_boxes, class_scores, nms_threshold)
        result_boxes.append(class_boxes[keep])
        result_scores.append(class_scores[keep])
        result_labels.append(torch.full((len(keep),), class_idx, dtype=torch.long))

    if len(result_boxes) == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    return torch.cat(result_boxes).cpu(), torch.cat(result_scores).cpu(), torch.cat(result_labels).cpu()

## テスト
評価指標には，物体検出タスクで広く使われる**mAP（mean Average Precision）**を用います．mAPの算出は`torchmetrics`の`MeanAveragePrecision`を利用します（詳細は`ssd.ipynb`を参照）．VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認します．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画します．

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l - 1]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナル論文との違い
本ノートブックで実装したFaster R-CNNは，原論文（およびResNetを用いたFaster R-CNN + ResNetの構成）を一部簡略化しています．主な違いは次の通りです．

| | 原論文（Faster R-CNN, ResNet-C4構成） | 本ノートブック |
|---|---|---|
| バックボーン | ResNet101（`conv1`〜`layer3`を共有特徴，`layer4`を第2段階に再利用） | ResNet50（同様の分割，ImageNet事前学習済み） |
| 入力画像サイズ | 短辺600ピクセル程度にリサイズ | 320×320に固定リサイズ |
| 学習方式 | 4-step alternating training（RPNとFast R-CNNを交互に学習） | RPN LossとROIヘッド Lossを1つの総損失として同時にend-to-endで学習 |
| ROIヘッドのボックス回帰 | クラスごとに異なる回帰（class-specific） | 全クラス共通の回帰（class-agnostic）に単純化 |
| RPN Proposal数（train pre/post-NMS） | 12000 / 2000 | 2000 / 1000 |
| RPN Proposal数（test pre/post-NMS） | 6000 / 300 | 2000 / 300 |
| NMS・IoU・ROIAlign | 独自実装 | `torchvision.ops`の`nms`・`box_iou`・`roi_align`を使用 |

また，学習初期はRPNのProposalがGTボックスとほとんど重ならないため，ROIヘッドの学習が進みにくいという問題があります．これに対して，原論文には明示されていませんが，多くの実装で採用されている実践的な工夫として，RPNのProposalに加えてGTボックス自体もROIヘッドのサンプリング候補に含めています．

## 課題

1. `RPN_POST_NMS_TOP_N`（推論時に残すProposal数）を変更し，検出精度（mAP）や推論時間にどのような影響があるか確認しましょう．

2. `ANCHOR_SCALES`や`ANCHOR_RATIOS`の構成を変更し，RPNが生成するアンカーの再現率（GTボックスをどれだけカバーできるか）や，最終的な検出精度への影響を確認しましょう．

3. `ROIHead`のボックス回帰を，class-agnostic（全クラス共通，4次元）からclass-specific（クラスごとに異なる，`n_class * 4`次元）に変更し，パラメータ数や検出精度への影響を確認しましょう．